# Floating Point: How Computers Store Numbers (and Where It Bites)

**This notebook teaches floating-point numbers from scratch, no prior exposure is assumed.** It is the companion to the Unit 1-5 slides and follows them section by section.

A computer has finite memory, so it can hold only finitely many numbers exactly; every other real number is replaced by the nearest one it has. That substitution is **rounding**, and almost always it is harmless: the error is about one part in $10^{16}$, far below the noise in any measurement you will ever feed a computer. This notebook shows how the machine writes a number down, exactly which numbers it can write, and the three *shapes* of computation (absorption, overflow and underflow, cancellation) where that tiny error gets amplified and arrives silently.

## Learning objectives

- Read a decimal or binary number by **place value**, and write one in **scientific notation** in either base.
- Write down every number in a small **floating-point system** $(\beta, p, L, U)$, and derive its four characteristic numbers: **UFL**, **OFL**, **machine epsilon**, and the **count**.
- Explain why machine epsilon is a **relative** step that applies at every magnitude.
- Recognize **absorption**, **overflow and underflow**, and **catastrophic cancellation** when they happen, and apply the one-line habit that avoids each.
- Say why, for ordinary arithmetic on ordinary data, floating point is **not** the accuracy bottleneck.

## Background

Positional notation (Unit 1-2): a digit's position says which power of the base it multiplies, so $253 = 2 \cdot 10^2 + 5 \cdot 10^1 + 3 \cdot 10^0$. Series and partial sums (Unit 1-4): the harmonic series $\sum 1/n$ diverges. Both are restated where they are used.

## This notebook covers

1. The problem: `0.1 + 0.2` is not `0.3`.
2. Place value in base 10 and base 2.
3. Scientific notation in any base.
4. The formal system, and a toy system written out in full.
5. Four numbers worth naming: UFL, OFL, machine epsilon, and the count.
6. Machine epsilon is a relative step, valid at every size.
7. Real systems: half, single, double, and beyond.
8. Absorption: a small number vanishes, and a divergent series freezes.
9. Overflow and underflow, and the fix of working in logs.
10. Cancellation: subtracting two close numbers.
11. Keep it in proportion.

**Prerequisites:** none, this notebook starts from scratch.

**Dataset:** none, numerical demonstrations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

## 1. The problem: `0.1 + 0.2` is not `0.3`

Type the two lines below into any language, Python, R, C, Java, or Excel, and you get the same answer. It is not a bug. The machine cannot store $0.1$ exactly, for the same reason you cannot write $\tfrac{1}{3} = 0.333\ldots$ in full: you write $0.333$ and stop, and have quietly written down a number that is *not* a third.

In [ ]:
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2        =", 0.1 + 0.2)

A number is stored in a fixed number of **bits**, 64 typically, so only **finitely many** numbers exist exactly. Every other real number is replaced by the nearest one the machine has; that substitution is **rounding**, and the difference is **rounding error**. The rest of the notebook makes that precise and then shows the few situations where it matters.

## 2. Place value: just add up the places

A digit's **position** says which power of the base it multiplies: $0$ at the units place, counting up to the left and *negative* to the right of the point. In base 10 you do this without thinking:

$$253.75_{10} = 2 \cdot 10^{2} + 5 \cdot 10^{1} + 3 \cdot 10^{0} + 7 \cdot 10^{-1} + 5 \cdot 10^{-2} = 200 + 50 + 3 + 0.7 + 0.05$$

Base 2 runs the **same scheme** with only the digits $0$ and $1$ (a transistor is reliably off or on), so the places are powers of 2:

$$11001.101_{2} = 1 \cdot 2^{4} + 1 \cdot 2^{3} + 0 \cdot 2^{2} + 0 \cdot 2^{1} + 1 \cdot 2^{0} + 1 \cdot 2^{-1} + 0 \cdot 2^{-2} + 1 \cdot 2^{-3} = 16 + 8 + 1 + \tfrac{1}{2} + \tfrac{1}{8} = 25.625_{10}$$

One function does both: walk along the digit string, and multiply each digit by its place's power of the base.

In [ ]:
def place_value(digits, base):
    '''Evaluate a digit string like '253.75' or '11001.101' in the given base, place by place.'''
    whole, _, frac = digits.partition('.')
    total = 0
    terms = []
    for position, d in enumerate(reversed(whole)):          # units place is position 0, counting up to the left
        total += int(d) * base**position
        terms.append(f"{d}*{base}^{position}")
    for position, d in enumerate(frac, start=1):            # right of the point, the exponents go negative
        total += int(d) * base**(-position)
        terms.append(f"{d}*{base}^-{position}")
    return total, " + ".join(reversed(terms[:len(whole)])) + " + " + " + ".join(terms[len(whole):])

for digits, base in [("253.75", 10), ("11001.101", 2)]:
    value, expansion = place_value(digits, base)
    print(f"{digits} (base {base}) = {expansion}")
    print(f"{'':>{len(digits) + 10}} = {value}\n")

Note which fractions base 2 can reach: **a binary fraction is a sum of powers of one half**, halves, quarters, eighths, and nothing else. That single fact is why $0.1$ is not exact. Converting the other way, from decimal to binary, uses two loops: repeated division by 2 for the whole part and repeated multiplication by 2 for the fraction. Run the fraction loop on $0.625$ and it finishes in three bits; run it on $0.1$ and it never finishes.

In [ ]:
def fraction_to_binary(frac, max_bits=20):
    '''Repeated multiplication by 2: the integer part that pops out each time is the next bit.'''
    bits = []
    for _ in range(max_bits):
        frac = frac * 2
        bit = int(frac)
        bits.append(str(bit))
        frac = frac - bit
        if frac == 0:                                       # nothing left: the expansion terminates
            return "0." + "".join(bits)
    return "0." + "".join(bits) + "..."                     # ran out of room: it repeats forever

print("0.625 in binary:", fraction_to_binary(0.625))
print("0.1   in binary:", fraction_to_binary(0.1))

## 3. Scientific notation, in any base

Scientists write a wide-ranging number as a modest number times a power of ten: $25.625 = +\,2.5625 \times 10^{1}$, with a **sign**, a **mantissa**, a **base**, and an **exponent**. A computer does exactly the same thing in base 2: $11001.101_2 = +\,1.1001101_2 \times 2^{4}$.

The point does not sit in a fixed place; the exponent **floats** it left or right, which is what lets 64 bits cover everything from the mass of a proton to the mass of a galaxy. Python will show you the machine's own scientific notation with `float.hex()`: the digits after `0x1.` are the mantissa in hexadecimal (each hex digit is four binary digits) and the number after `p` is the binary exponent.

In [ ]:
x = 25.625
print(f"{x} in the machine's notation: {x.hex()}")

# unpack that: the hex mantissa 9a is binary 1001 1010, so 0x1.9a = 1.10011010 in binary, times 2^4
mantissa_hex = x.hex().split('.')[1].split('p')[0].rstrip('0')
mantissa_bits = "".join(f"{int(h, 16):04b}" for h in mantissa_hex).rstrip('0')
exponent = int(x.hex().split('p')[1])
print(f"mantissa 1.{mantissa_bits} (binary), exponent {exponent}: check {(1 + int(mantissa_bits, 2) / 2**len(mantissa_bits)) * 2**exponent}")

## 4. The formal system, and a toy system written out in full

Every representable number in a floating-point system has the form

$$x = \pm \left( d_0 + \frac{d_1}{\beta} + \frac{d_2}{\beta^2} + \cdots + \frac{d_{p-1}}{\beta^{p-1}} \right) \beta^{E}$$

where

- $\beta$ is the **base** ($2$ for a computer, $10$ for us);
- $p$ is the **precision**, the number of digits kept;
- $d_0, d_1, \ldots, d_{p-1}$ are those digits, the **mantissa**, each with $0 \le d_i \le \beta - 1$;
- $E$ is the **exponent**, an integer restricted to a range $L \le E \le U$.

Four integers, $\beta, p, L, U$, define the whole system. Most systems are **normalized**, $d_0 \ne 0$; in base 2 that forces $d_0 = 1$, so the leading bit need not be stored (a free extra digit) and every value has exactly one spelling.

### 4.1 A toy system

Take $\beta = 2$, $p = 3$, $L = -1$, $U = 1$. A normalized mantissa is $(1.d_1 d_2)_2$, so there are four of them, and three exponents. The loop below generates **every** number in the system straight from the definition: a sign, a mantissa, an exponent.

In [ ]:
beta, p, L, U = 2, 3, -1, 1

toy = []
for sign in (+1, -1):
    for E in range(L, U + 1):
        for d1 in range(beta):                              # d0 is forced to 1 by normalization
            for d2 in range(beta):
                mantissa = 1 + d1 / beta + d2 / beta**2
                toy.append(sign * mantissa * beta**E)
toy.append(0.0)                                             # zero cannot be spelled in normalized form, so it is added by hand
toy = sorted(set(toy))

print(f"the toy system has {len(toy)} numbers:")
print(toy)

The same numbers as a table, one row per exponent and one column per mantissa, so you can see how the exponent scales a whole row.

In [ ]:
mantissas = {f"1.{d1}{d2} = {1 + d1/beta + d2/beta**2}": 1 + d1/beta + d2/beta**2 for d1 in range(beta) for d2 in range(beta)}
table = pd.DataFrame({name: [f"±{m * beta**E:g}" for E in range(L, U + 1)] for name, m in mantissas.items()},
                     index=[f"E = {E}, times {beta}^{E}" for E in range(L, U + 1)])
table

That is the entire system. Nothing else exists: $0.3$ does not, $4$ does not, $0.1$ does not. On a number line the pattern is the point: within a row the mantissa steps evenly, but the exponent scales the row, so **the gaps double from one exponent to the next**, and the numbers are dense near zero and sparse out at the edges.

In [ ]:
# --- graphics: the toy system on a number line (mechanics only) ---
fig, ax = plt.subplots(figsize=(9, 1.8))
ax.scatter(toy, np.zeros_like(toy), s=30, color="tab:cyan", zorder=3)
ax.axhline(0, color="gray", lw=1)
ax.set_yticks([])
ax.set_xlabel("value")
ax.set_title(f"every number in the toy system (beta={beta}, p={p}, L={L}, U={U}): {len(toy)} of them")
for spine in ("left", "right", "top"):
    ax.spines[spine].set_visible(False)
plt.show()

## 5. Four numbers worth naming

Each is read straight off the table: pick the mantissa and the exponent that produce it.

- **UFL**, the underflow limit, the smallest positive number: smallest mantissa $1.00\ldots0 = 1$ times the smallest exponent, $\text{UFL} = \beta^{L}$.
- **OFL**, the overflow limit, the largest number: largest mantissa (every digit $\beta - 1$, which sums to $\beta - \beta^{1-p}$) times the largest exponent, $\text{OFL} = (\beta - \beta^{1-p})\,\beta^{U} = \beta^{U+1}(1 - \beta^{-p})$.
- **Machine epsilon**, the gap just above $1$: one unit in the last mantissa digit, at the exponent of $1$, $\varepsilon = \beta^{-(p-1)} \cdot \beta^{0} = \beta^{1-p}$.
- **The count**, built the way the table was built: $2$ signs, $(\beta - 1)$ choices for $d_0 \ne 0$, $\beta^{p-1}$ choices for the remaining digits, $(U - L + 1)$ exponents, plus $1$ for zero.

The cell computes all four from the formulas and checks each one against the list of numbers we generated.

In [ ]:
UFL   = beta**L
OFL   = beta**(U + 1) * (1 - beta**(-p))
eps   = beta**(1 - p)
count = 2 * (beta - 1) * beta**(p - 1) * (U - L + 1) + 1

positives = [t for t in toy if t > 0]
gap_above_one = min(t for t in toy if t > 1) - 1

print(f"UFL     formula {UFL:<6} from the list {min(positives):<6}  match: {UFL == min(positives)}")
print(f"OFL     formula {OFL:<6} from the list {max(toy):<6}  match: {OFL == max(toy)}")
print(f"epsilon formula {eps:<6} from the list {gap_above_one:<6}  match: {eps == gap_above_one}")
print(f"count   formula {count:<6} from the list {len(toy):<6}  match: {count == len(toy)}")

The same four numbers for the machine you are actually using. NumPy reports them with `np.finfo`: `tiny` is UFL, `max` is OFL, `eps` is machine epsilon, and `nmant` is the number of stored mantissa bits (the free leading 1 makes $p$ one larger).

In [ ]:
info = np.finfo(np.float64)
p_double = info.nmant + 1                                   # stored bits plus the free leading 1

print(f"double precision: p = {p_double} bits, about {info.precision} decimal digits")
print(f"UFL     = {info.tiny}")
print(f"OFL     = {info.max}")
print(f"epsilon = {info.eps}   (check: 2^(1 - p) = {2.0**(1 - p_double)})")
print(f"exponent range L = {info.minexp}, U = {info.maxexp - 1}")

## 6. Machine epsilon is a relative step, valid at every size

**Machine epsilon** $\varepsilon = \beta^{1-p}$ is the gap between $1$ and the next representable number: the smallest $\varepsilon$ with $\text{fl}(1 + \varepsilon) > 1$. In the toy system that is $0.25$, the gap from $1$ to $1.25$; in a double it is $2^{-52}$.

In [ ]:
eps = np.finfo(float).eps
print("machine epsilon =", eps)
print("1 + eps   > 1 ?", 1.0 + eps > 1.0)                   # just big enough to register
print("1 + eps/2 > 1 ?", 1.0 + eps / 2 > 1.0)               # too small: rounds back to 1

It is not really about $1$. The gap just above any number $x = m \cdot \beta^{E}$ is $\varepsilon \cdot \beta^{E}$, so the gaps **scale with the number**: in the toy system $0.125$ near $0.5$, $0.25$ near $1$, $0.5$ near $2$, always about $\varepsilon \cdot x$. So $\varepsilon$ is **the smallest relative amount you can add to a number and have it change**: $x + y$ moves off $x$ only when $|y| \gtrsim |x| \cdot \varepsilon / 2$, whether $x$ is $1$ or $10^{20}$.

`np.spacing(x)` returns the gap just above $x$. Divide it by $x$ and you get roughly $\varepsilon$ at every magnitude.

In [ ]:
print(f"{'x':>8} {'gap above x':>14} {'gap / x':>12}")
for x in [0.5, 1.0, 2.0, 1e3, 1e10, 1e20]:
    print(f"{x:8.0e} {np.spacing(x):14.3e} {np.spacing(x) / x:12.3e}")
print(f"\nmachine epsilon for comparison: {eps:.3e}")

Read as an error bound: rounding costs at most half a gap, so $|\text{fl}(x) - x| / |x| \le \varepsilon / 2$. You get about 16 **significant digits** wherever you stand, not 16 decimal places.

## 7. Real systems: what the bits buy you

Every real machine is the same $(\beta = 2, p, L, U)$ recipe with a different bit budget. The IEEE 754 **double**, the default `float` in Python, NumPy, R and MATLAB, spends 64 bits as 1 sign bit, 11 exponent bits, and 52 mantissa bits (plus the free leading 1, so $p = 53$). NumPy also carries the smaller formats used in graphics and machine learning; `np.finfo` describes each one.

In [ ]:
rows = []
for name, dtype in [("half", np.float16), ("single", np.float32), ("double", np.float64), ("long double", np.longdouble)]:
    fi = np.finfo(dtype)
    rows.append({"format": name, "bits": fi.bits, "p (mantissa bits + 1)": fi.nmant + 1,
                 "decimal digits": fi.precision, "epsilon": f"{float(fi.eps):.2e}",
                 "UFL": f"{float(fi.tiny):.2e}", "OFL": f"{float(fi.max):.2e}"})
formats = pd.DataFrame(rows).set_index("format")
formats

Two things to notice. Nobody agreed on a format until IEEE 754 in 1985, so the same program used to give different answers on different machines. And the trend is not upward: machine learning now runs on 16-bit formats with 2 to 3 digits, trading precision for speed, which is exactly why the failures below are back. (On Windows, `np.longdouble` is usually just another name for the 64-bit double, so its row may match the double's.)

## 8. Absorption: the small number disappears

The true sum $10^{20} + 10^{-20}$ needs 40 significant digits; a double keeps 16, so everything past the sixteenth is rounded away and the larger number survives unchanged.

**The rule:** adding $y$ to $x$ changes nothing whenever $|y| < |x| \cdot \varepsilon / 2$. The small term falls inside the gap around the large one, so the nearest representable number is still $x$.

In [ ]:
print("1e20 + 1e-20 == 1e20 ?", 1e20 + 1e-20 == 1e20)

x = 1e20
threshold = x * eps / 2
print(f"anything below about {threshold:.2e} is absorbed by {x:.0e}")
print(f"  {x:.0e} + {threshold / 10:.1e}  changes it? {x + threshold / 10 != x}")
print(f"  {x:.0e} + {threshold * 10:.1e}  changes it? {x + threshold * 10 != x}")

### 8.1 A divergent series that stops growing

Run the harmonic series from Unit 1-4 in a loop, adding one term at a time to a running total $S_n$:

$$S_n = \sum_{k=1}^{n} \frac{1}{k} = 1 + \frac{1}{2} + \frac{1}{3} + \cdots + \frac{1}{n}, \qquad \text{which diverges: } S_n \to \infty.$$

On a computer the terms shrink while the total grows, so eventually $\tfrac{1}{n} < S_n \cdot \varepsilon / 2$. From then on every term is absorbed, $S_{n+1} = S_n$ exactly, and the partial sums **freeze**. To see it in seconds we use 32-bit floats, whose $\varepsilon$ is larger; a double freezes the same way, just after far more terms. The loop stops the moment adding a term changes nothing, and then checks the rule.

In [ ]:
eps32 = np.finfo(np.float32).eps

total = np.float32(0.0)
n = 0
snapshots = []                                              # (n, total) every so often, for the plot
while True:
    n += 1
    term = np.float32(1.0) / np.float32(n)
    new_total = total + term
    if new_total == total:                                  # the term was absorbed: the sum has frozen
        break
    total = new_total
    if n % 5000 == 0:
        snapshots.append((n, float(total)))

print(f"the harmonic sum froze at n = {n}")
print(f"frozen value S = {float(total):.6f}   (mathematically it should grow without bound)")
print(f"check the rule: 1/n = {float(term):.3e}  versus  S * eps/2 = {float(total) * eps32 / 2:.3e}")

In [ ]:
# --- graphics: the running total climbs, then goes flat (mechanics only) ---
ns, totals = zip(*snapshots)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, totals, color="tab:cyan")
ax.set_xlabel("number of terms n")
ax.set_ylabel("running total S_n")
ax.set_title("the harmonic sum in 32-bit floats: it stops growing")
plt.show()

The divergent series *looks* convergent, and no error is raised. The habit: when a sum spans many magnitudes, add the small terms first, or use `math.fsum` / `np.sum`, which are written to lose as little as possible.

## 9. Overflow and underflow, and the fix of working in logs

Ask for a number past OFL and you get `inf`; ask for one below UFL and you get `0`, both silently. `np.exp(1000)` is about $10^{434}$, past the largest double; `np.exp(-1000)` is a tiny positive number that becomes exactly zero, after which $\ln 0 = -\infty$ and any division by it explodes.

In [ ]:
with np.errstate(over='ignore', divide='ignore'):
    print("np.exp(1000)  =", np.exp(1000))
    print("np.exp(-1000) =", np.exp(-1000))
    print("np.log(np.exp(-1000)) =", np.log(np.exp(-1000)))

The everyday version is a **long product**. Multiplying 2000 probabilities, each a little under $1$, underflows to exactly $0$. The classic fix is to take logarithms, which turns the product into a sum (Unit 1-4) and the magnitudes back into something a double can hold:

$$\prod_{i=1}^{n} p_i \quad \longrightarrow \quad \sum_{i=1}^{n} \ln p_i$$

That is why statistics maximizes the **log**-likelihood, and why classifiers expose `predict_log_proba`: a numerical decision, not a mathematical one.

In [ ]:
rng = np.random.default_rng(0)
probs = rng.uniform(0.2, 0.9, size=2000)                    # 2000 probabilities, each comfortably above zero

product = np.prod(probs)
log_sum = np.sum(np.log(probs))

print(f"product of {len(probs)} probabilities directly: {product}")
print(f"sum of their logs:                       {log_sum:.3f}   (so the product is about 10^{log_sum / np.log(10):.0f})")

## 10. Cancellation: subtracting two close numbers

Subtracting nearly equal numbers is the most destructive operation in numerical computing. Watch it on a **decimal machine that keeps $p = 4$ significant digits**, so the effect is visible by eye. Two inputs, each stored about as well as such a machine can store anything, are subtracted; their leading digits are identical, so they cancel to zero, and cancelling carries no information. What is left is the last digit of each, exactly where the rounding error lived. The error did not grow; the *result* shrank, so the error became a much larger *share* of it.

In [ ]:
def store(x, digits=4):
    '''Round x to the given number of significant digits: what a p-digit decimal machine keeps.'''
    if x == 0:
        return 0.0
    magnitude = math.floor(math.log10(abs(x)))
    return round(x, digits - 1 - magnitude)

a, b = 1.23456, 1.23444
fa, fb = store(a), store(b)

rows = [("a", a, fa), ("b", b, fb), ("a - b", a - b, fa - fb)]
print(f"{'':>6} {'true value':>12} {'stored (4 digits)':>18} {'relative error':>15}")
for name, true, stored in rows:
    print(f"{name:>6} {true:12.5f} {stored:18.5f} {abs(stored - true) / abs(true):15.2%}")

A 16-digit double does the same thing, just further to the right. This is **catastrophic cancellation**, and it is a property of the *formula*, not of the data; rearranging the algebra usually avoids it.

### 10.1 Where it hides in data science: a naive variance

The textbook shortcut $\text{Var}(x) = \overline{x^2} - \bar{x}^{2}$ subtracts two nearly equal numbers whenever the data sit far from zero. Here the values are around $10^{8}$ with a spread of about $1$, so both terms are about $10^{16}$ and the true variance is about $1$: the whole answer lives in the last digit of each. The library formula, which centres the data first, has no such subtraction.

In [ ]:
rng = np.random.default_rng(1)
x = 1e8 + rng.normal(0, 1, size=100_000)                    # values near 1e8, spread about 1

naive_var = np.mean(x**2) - np.mean(x)**2                   # two near-equal terms of size 1e16, subtracted
library_var = np.var(x)                                     # centres the data first: no near-equal subtraction

print(f"naive   mean(x^2) - mean(x)^2 = {naive_var:.6f}")
print(f"library np.var(x)             = {library_var:.6f}")
print(f"true variance (by construction): about 1")

The naive result can even come out *negative*, which a variance never is. Same mathematics, different arithmetic, different accuracy. The habit: rearrange the formula, or use the library version, which already has.

## 11. Keep it in proportion

Every failure above was an *algorithm* amplifying a tiny error, and only a few shapes do it. For ordinary arithmetic on ordinary data the double is far better than the data: a lab assay is good to 3 digits, a survey answer to 1 or 2, a clinical measurement rarely to 4, and the double carries 16. The cell below computes the mean of a thousand measurements in ordinary floating point and again in **exact rational arithmetic** (Python's `Fraction` never rounds), then compares the two. The disagreement is the entire effect of floating point on the calculation; the measurement noise is shown beside it.

In [ ]:
from fractions import Fraction

rng = np.random.default_rng(2)
measurements = np.round(rng.normal(120, 15, size=1000), 1)  # blood pressures recorded to 0.1 mmHg

float_mean = measurements.mean()
exact_mean = sum(Fraction(str(v)) for v in measurements) / len(measurements)

rounding_error = abs(Fraction(str(float_mean)) - exact_mean) / exact_mean
measurement_error = 0.05 / float(exact_mean)                # half the last recorded digit, relative to the mean

print(f"floating-point mean      = {float_mean}")
print(f"exact rational mean      = {float(exact_mean)}")
print(f"relative rounding error  = {float(rounding_error):.1e}")
print(f"relative measurement err = {measurement_error:.1e}   ({measurement_error / max(float(rounding_error), 1e-300):.0e} times larger)")

The **data**, not the arithmetic, is the accuracy bottleneck. If none of the shapes below is in your code, floating point is not your problem. The aim is to be calm, not careful: check for these, then trust the machine.

| the shape | the habit |
|---|---|
| a sum spanning many magnitudes | `math.fsum` or `np.sum`, small terms first |
| a long product | work in logs: `logsumexp`, log-likelihoods |
| subtracting nearly equal numbers | rearrange the formula, or use the library version (`np.var`) |
| exact quantities: money, counts | integers, not floats |

## 12. Summary

- A digit's position says which power of the base it multiplies; binary is the same scheme with digits 0 and 1, so binary fractions are sums of powers of one half, and $0.1$ is stored a hair off. That is all `0.1 + 0.2 != 0.3` means.
- A floating-point system is scientific notation in base $\beta$ with $p$ digits and exponents $L \le E \le U$; the toy system $(2, 3, -1, 1)$ has exactly 25 numbers, and the gaps double with every exponent.
- Its four numbers: $\text{UFL} = \beta^{L}$, $\text{OFL} = \beta^{U+1}(1 - \beta^{-p})$, $\varepsilon = \beta^{1-p}$, and the count $2(\beta - 1)\beta^{p-1}(U - L + 1) + 1$.
- **Machine epsilon** is a relative step at every magnitude: the gap above $x$ is about $\varepsilon x$, so adding less than $|x|\,\varepsilon/2$ changes nothing.
- **Absorption** freezes a sum spanning many magnitudes; **overflow and underflow** live past OFL and below UFL, and long products are handled in logs; **cancellation** throws away the digits that agreed and leaves the answer resting on rounding error.
- For ordinary arithmetic on ordinary data the double is far better than the data, so know the shapes, apply the habit, and trust the machine.

Next: linear algebra, vectors, matrices, and the systems they solve (`U2-1_Systems-1_GeometryOfSolutions`).